# SHAP Values Practice: Voter Data

Special topic companion notebook for lecture 30. Covers SHAP (SHapley Additive exPlanations) applied to bagging, random forests, and logistic regression.

## Setup

*Same voter data setup as notebooks 18–20.*

> If `shap` is not installed, run: `pip install shap`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import warnings

warnings.filterwarnings('ignore')
np.random.seed(4025)

# Load and clean
voter_url = 'https://raw.githubusercontent.com/fivethirtyeight/data/master/non-voters/nonvoters_data.csv'
voter_raw = pd.read_csv(voter_url)

voter_clean = voter_raw.drop(columns=['RespId', 'weight', 'Q1']).copy()
voter_clean['educ'] = pd.Categorical(voter_clean['educ'],
    categories=['High school or less', 'Some college', 'College'], ordered=True)
voter_clean['income_cat'] = pd.Categorical(voter_clean['income_cat'],
    categories=['Less than $40k', '$40-75k ', '$75-125k', '$125k or more'], ordered=True)
voter_clean['voter_category'] = pd.Categorical(voter_clean['voter_category'],
    categories=['rarely/never', 'sporadic', 'always'], ordered=False)

voter_clean = voter_clean[voter_clean['Q22'] != 5].copy()
voter_clean['Q22'] = voter_clean['Q22'].fillna('Not Asked').astype(str)

for c in [f'Q28_{i}' for i in range(1, 9)]:
    voter_clean[c] = voter_clean[c].replace(-1.0, 0).fillna('Not Asked').astype(str)
for c in [f'Q29_{i}' for i in range(1, 11)]:
    voter_clean[c] = voter_clean[c].replace(-1.0, 0).fillna('Not Asked').astype(str)

def _party_id(row):
    q31, q32, q33 = row['Q31'], row['Q32'], row['Q33']
    if   q31 == 1: return 'Strong Republican'
    elif q31 == 2: return 'Republican'
    elif q32 == 1: return 'Strong Democrat'
    elif q32 == 2: return 'Democrat'
    elif q33 == 1: return 'Lean Republican'
    elif q33 == 2: return 'Lean Democrat'
    else:          return 'Other'

voter_clean['Party_ID'] = voter_clean.apply(_party_id, axis=1)
party_order = ['Strong Republican', 'Republican', 'Lean Republican',
               'Other', 'Lean Democrat', 'Democrat', 'Strong Democrat']
voter_clean['Party_ID'] = pd.Categorical(voter_clean['Party_ID'],
                                         categories=party_order, ordered=True)
voter_clean = voter_clean.drop(columns=['Q31', 'Q32', 'Q33'])

for c in voter_clean.columns:
    if c != 'ppage' and voter_clean[c].dtype in ['float64', 'int64']:
        voter_clean[c] = voter_clean[c].replace(-1, np.nan)

X_voter = voter_clean.drop(columns=['voter_category'])
y_voter = voter_clean['voter_category']

X_train, X_test, y_train, y_test = train_test_split(
    X_voter, y_voter, test_size=0.3, stratify=y_voter, random_state=4025)

ordered_cat_cols = ['educ', 'income_cat', 'Party_ID']
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
nom_cols = [c for c in X_train.columns
            if c not in ordered_cat_cols and c not in num_cols]

preproc_voter = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median'))]), num_cols),
    ('ord', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]), ordered_cat_cols),
    ('nom', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
    ]), nom_cols),
], remainder='drop')

print(f"Training: {X_train.shape[0]} obs  |  Test: {X_test.shape[0]} obs")

## Fit Models

In [ ]:
bag_pipe = Pipeline([
    ('preproc', preproc_voter),
    ('bag', BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=4025),
        n_estimators=200, oob_score=True, random_state=4025, n_jobs=-1
    ))
])
bag_pipe.fit(X_train, y_train)

rf_pipe = Pipeline([
    ('preproc', preproc_voter),
    ('rf', RandomForestClassifier(
        n_estimators=200, max_features='sqrt', random_state=4025, n_jobs=-1
    ))
])
rf_pipe.fit(X_train, y_train)

lr_pipe = Pipeline([
    ('preproc', preproc_voter),
    ('lr', LogisticRegression(max_iter=1000, random_state=4025))
])
lr_pipe.fit(X_train, y_train)

# Feature names after encoding
enc = rf_pipe.named_steps['preproc'].named_transformers_['nom'].named_steps['encode']
feature_names = num_cols + ordered_cat_cols + list(enc.get_feature_names_out(nom_cols))

for name, pipe in [('Bagging', bag_pipe), ('Random Forest', rf_pipe), ('Logistic Reg.', lr_pipe)]:
    print(f"{name:15s}  test acc = {accuracy_score(y_test, pipe.predict(X_test)):.3f}")
print(f"\n{len(feature_names)} features after encoding")

## SHAP Values

### SHAP: Preprocess Data

SHAP operates on the model's internal estimator (not the full pipeline), so we run the preprocessing step manually to get transformed DataFrames with column names.

In [ ]:
X_train_proc = pd.DataFrame(
    rf_pipe.named_steps['preproc'].transform(X_train),
    columns=feature_names
)
X_test_proc = pd.DataFrame(
    rf_pipe.named_steps['preproc'].transform(X_test),
    columns=feature_names
)

# Use a subset of 300 test observations for speed
X_test_sub = X_test_proc.iloc[:300]

# Class labels (matches order in y_voter.cat.categories)
class_names = list(y_voter.cat.categories)  # ['rarely/never', 'sporadic', 'always']
always_idx = 2  # index of 'always' class
print("Classes:", class_names)

### SHAP: Random Forest — Compute

`TreeExplainer` exploits the tree structure for exact, fast SHAP computation. For a 3-class problem, `shap_vals_rf.values` has shape `(n_obs, n_features, 3)` — one set of SHAP values per class.

In [ ]:
explainer_rf = shap.TreeExplainer(rf_pipe.named_steps['rf'])
shap_vals_rf = explainer_rf(X_test_sub)

print("SHAP values shape:", shap_vals_rf.values.shape)
print("Base values shape:", shap_vals_rf.base_values.shape)
print(f"Base value for '{class_names[always_idx]}': {shap_vals_rf.base_values[0, always_idx]:.3f}")

### SHAP: Global Bar Plot (Random Forest)

Mean $|\phi_j|$ across all 300 observations for the `'always'` class — a global view of which features most affect the predicted probability of being an always-voter.

In [ ]:
shap.plots.bar(shap_vals_rf[:, :, always_idx], max_display=15, show=True)

### SHAP: Beeswarm Plot (Random Forest)

Each dot is one observation. The x-axis shows the SHAP value (positive = pushes toward `'always'`). Color encodes the feature value: red = high, blue = low.

This reveals not just *which* features matter, but *how* — direction and heterogeneity of effects.

In [ ]:
shap.plots.beeswarm(shap_vals_rf[:, :, always_idx], max_display=15, show=True)

### SHAP: Local Explanation (Waterfall Plot)

A waterfall plot explains a **single prediction**. It starts at the base value (the model's average predicted probability for the `'always'` class) and each feature's SHAP value adds or subtracts until reaching the actual predicted probability for this observation.

Red bars push the prediction up; blue bars push it down.

In [ ]:
# Explain the first test observation's predicted probability for 'always'
obs_idx = 0
shap.plots.waterfall(shap_vals_rf[obs_idx, :, always_idx], max_display=12, show=True)

### SHAP: Logistic Regression

`LinearExplainer` computes exact SHAP values analytically for linear models — very fast. We pass the training data as the background distribution.

In [ ]:
explainer_lr = shap.LinearExplainer(
    lr_pipe.named_steps['lr'],
    X_train_proc
)
shap_vals_lr = explainer_lr(X_test_sub)

print("SHAP values shape:", shap_vals_lr.values.shape)

shap.plots.bar(shap_vals_lr[:, :, always_idx], max_display=15, show=True)

### SHAP: Compare RF vs. Logistic Regression

Side-by-side mean $|\phi_j|$ for the `'always'` class across both models. Features that rank highly for both are the most trustworthy signals.

In [ ]:
rf_shap_mean = np.abs(shap_vals_rf.values[:, :, always_idx]).mean(axis=0)
lr_shap_mean = np.abs(shap_vals_lr.values[:, :, always_idx]).mean(axis=0)

shap_compare = (pd.DataFrame({
    'feature':       feature_names,
    'Random Forest': rf_shap_mean,
    'Logistic Reg.': lr_shap_mean,
})
.sort_values('Random Forest', ascending=False)
.head(15))

fig, ax = plt.subplots(figsize=(8, 6))
x = np.arange(len(shap_compare))
w = 0.35
ax.barh(x - w/2, shap_compare['Random Forest'][::-1].values, w,
        label='Random Forest', color='darkorange')
ax.barh(x + w/2, shap_compare['Logistic Reg.'][::-1].values, w,
        label='Logistic Reg.', color='seagreen')
ax.set_yticks(x)
ax.set_yticklabels(shap_compare['feature'][::-1].values)
ax.set_xlabel("Mean |SHAP value| for 'always' class")
ax.set_title('SHAP Importance: Random Forest vs. Logistic Regression')
ax.legend()
plt.tight_layout()
plt.show()